In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)
from dotenv import load_dotenv

project_root = os.path.abspath('..')
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, 'src'))  # 让 Python 能直接找到 src 下的模块
from src.models import FaceAttributeModel
from src.config import TARGET_ATTRS

load_dotenv()


PROCESSED_DIR = os.getenv("PROCESSED_DATA_DIR")
MODEL_SAVE_PATH = os.getenv("MODEL_SAVE_PATH", "./models/best_model.pth")
DEVICE = os.getenv("DEVICE", "cpu")

print("✅ 库导入完成")
print(f"📁 处理数据目录: {PROCESSED_DIR}")
print(f"⚙️  运行设备: {DEVICE}")


✅ 库导入完成
📁 处理数据目录: E:/BaiDuwp/Dataset/processed
⚙️  运行设备: cpu


In [2]:

test_features_path = os.path.join(project_root, "data", "processed", "celeba_test_features.npz")
if not os.path.exists(test_features_path):
    raise FileNotFoundError(
        f"❌ 测试集特征不存在：{test_features_path}")

data = np.load(test_features_path)
X_test = data['features']
gender_test = data['gender']
print(f"✅ 测试集特征加载完成: {X_test.shape}")


✅ 测试集特征加载完成: (39812, 512)


In [3]:
# ==================== 加载测试集属性标签（快速修复版） ====================
import pandas as pd
import os
import numpy as np
from dotenv import load_dotenv
load_dotenv()

# 1. 读取属性文件（注意环境变量名是 ATTR_FILE，不是 ATTTR_FILE）
attr_file = os.getenv("ATTR_FILE")
if not attr_file:
    raw_dir = os.getenv("RAW_DATA_DIR")
    if raw_dir:
        attr_file = os.path.join(os.path.dirname(raw_dir), "list_attr_celeba.txt")
    else:
        raise ValueError("请在 .env 中设置 ATTR_FILE 或 RAW_DATA_DIR")

print(f"📄 属性文件路径: {attr_file}")

# 2. 检查文件是否存在
if not os.path.exists(attr_file):
    raise FileNotFoundError(f"属性文件不存在: {attr_file}")

# 3. 读取属性文件：第1行是数量，第2行是列名，第3行开始是数据
attr_df = pd.read_csv(attr_file, sep=r'\s+', skiprows=2, header=None)

# 4. 提取列名
with open(attr_file, 'r') as f:
    lines = f.readlines()
header_line = lines[1].strip().split()
attr_names = ['filename'] + header_line
attr_df.columns = attr_names

print(f"✅ 属性文件列数: {len(attr_df.columns)}")

# 5. 读取测试集文件名列表
test_csv = os.path.join(os.getenv("PROCESSED_DATA_DIR"), "test_images.csv")
if not os.path.exists(test_csv):
    raise FileNotFoundError(f"测试集CSV不存在: {test_csv}")

test_df = pd.read_csv(test_csv)
test_filenames = test_df['filename'].tolist()
print(f"✅ 测试集文件名数量: {len(test_filenames)}")

# 6. 筛选测试集
attr_df_test = attr_df[attr_df['filename'].isin(test_filenames)]
print(f"✅ 匹配到的测试集图片数量: {len(attr_df_test)}")

# 7. 对齐 X_test（如果 X_test 已加载）
try:
    if len(attr_df_test) > X_test.shape[0]:
        attr_df_test = attr_df_test.iloc[:X_test.shape[0]]
    elif len(attr_df_test) < X_test.shape[0]:
        print(f"⚠️ 标签数量 ({len(attr_df_test)}) 少于特征数量 ({X_test.shape[0]})")
        X_test = X_test[:len(attr_df_test)]
except NameError:
    print("⚠️ X_test 未定义，请先运行加载测试集特征的单元格")

# 8. 提取 14 个核心属性
y_test_attrs = attr_df_test[TARGET_ATTRS].values.astype(np.float32)

# 9. 将 -1/1 转为 0/1
for i in range(y_test_attrs.shape[1]):
    y_test_attrs[:, i] = (y_test_attrs[:, i] == 1).astype(float)

print(f"✅ 测试集属性标签加载完成: {y_test_attrs.shape}")
print(f"   Male 正样本数: {y_test_attrs[:, 0].sum():.0f}")
print(f"   Oval_Face 正样本数: {y_test_attrs[:, 1].sum():.0f}")

📄 属性文件路径: E:/BaiDuwp/Dataset/list_attr_celeba.txt
✅ 属性文件列数: 41
✅ 测试集文件名数量: 40685
✅ 匹配到的测试集图片数量: 40685
✅ 测试集属性标签加载完成: (39812, 14)
   Male 正样本数: 17115
   Oval_Face 正样本数: 10931


In [11]:
# 用 project_root 拼接正确的模型路径
model_path = os.path.join(project_root, "models", "best_model.pth")
print(f"模型路径: {model_path}")

if not os.path.exists(model_path):
    raise FileNotFoundError(f"模型文件不存在: {model_path}")

model = FaceAttributeModel().to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))  # 修正括号位置
model.eval()

# 拼接性别
genders = y_test_attrs[:, 0].reshape(-1, 1)
X_test_input = np.hstack([X_test, genders])

X_tensor = torch.tensor(X_test_input, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    logits = model(X_tensor)
    probs = torch.sigmoid(logits).cpu().numpy()
    preds = (probs > 0.5).astype(int)

print("✅ 预测完成")

模型路径: d:\Fightting_my_work\makeup-recommendation-project\models\best_model.pth
✅ 预测完成


In [13]:
print("=" * 60)
print("📈 各属性评估指标")
print("=" * 60)

metrics_list = []
for i, attr in enumerate(TARGET_ATTRS):  # 修正拼写：enumrate → enumerate
    y_true = y_test_attrs[:, i]
    y_pred = preds[:, i]
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    metrics_list.append({
        '属性': attr,
        '准确率': f"{acc:.3f}",
        '精确率': f"{prec:.3f}",
        '召回率': f"{rec:.3f}",
        'F1': f"{f1:.3f}"
    })

metrics_df = pd.DataFrame(metrics_list)
print(metrics_df.to_string(index=False))

print("\n📊 宏平均 (Macro Average):")
print(f"  准确率: {np.mean([float(m['准确率']) for m in metrics_list]):.3f}")
print(f"  精确率: {np.mean([float(m['精确率']) for m in metrics_list]):.3f}")
print(f"  召回率: {np.mean([float(m['召回率']) for m in metrics_list]):.3f}")
print(f"  F1分数: {np.mean([float(m['F1']) for m in metrics_list]):.3f}")

print("加载的模型路径:", MODEL_SAVE_PATH)
print("模型文件修改时间:", os.path.getmtime(MODEL_SAVE_PATH))

📈 各属性评估指标
                 属性   准确率   精确率   召回率    F1
               Male 1.000 1.000 1.000 1.000
          Oval_Face 0.725 0.000 0.000 0.000
             Chubby 0.951 0.000 0.000 0.000
    High_Cheekbones 0.555 0.000 0.000 0.000
        Double_Chin 0.960 0.000 0.000 0.000
        Narrow_Eyes 0.884 0.000 0.000 0.000
    Arched_Eyebrows 0.741 0.000 0.000 0.000
     Bushy_Eyebrows 0.742 0.070 0.060 0.065
           Big_Nose 0.695 0.414 0.770 0.539
        Pointy_Nose 0.726 0.000 0.000 0.000
           Big_Lips 0.534 0.304 0.717 0.427
Mouth_Slightly_Open 0.524 0.000 0.000 0.000
          Pale_Skin 0.953 0.000 0.000 0.000
              Young 0.227 0.000 0.000 0.000

📊 宏平均 (Macro Average):
  准确率: 0.730
  精确率: 0.128
  召回率: 0.182
  F1分数: 0.145
加载的模型路径: ./models/best_model.pth


FileNotFoundError: [WinError 3] 系统找不到指定的路径。: './models/best_model.pth'

In [ ]:
# 创建保存目录
os.makedirs("./evaluation_results", exist_ok=True)

# 6.1 混淆矩阵 - Male
plt.figure(figsize=(5, 4))
cm = confusion_matrix(y_test_attrs[:, 0], preds[:, 0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Female', 'Male'],
            yticklabels=['Female', 'Male'])
plt.title('混淆矩阵 - Male 性别')
plt.tight_layout()
plt.savefig("./evaluation_results/cm_male.png")
plt.show()

# 6.2 ROC 曲线 - Male
plt.figure(figsize=(5, 4))
fpr, tpr, _ = roc_curve(y_test_attrs[:, 0], probs[:, 0])
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'Male (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('假阳性率 (FPR)')
plt.ylabel('真阳性率 (TPR)')
plt.title('ROC 曲线 - Male')
plt.legend()
plt.tight_layout()
plt.savefig("./evaluation_results/roc_male.png")
plt.show()

# 6.3 所有属性柱状图对比
plt.figure(figsize=(12, 5))
x = np.arange(len(TARGET_ATTRS))
width = 0.25
acc_vals = [float(m['准确率']) for m in metrics_list]
rec_vals = [float(m['召回率']) for m in metrics_list]
f1_vals = [float(m['F1']) for m in metrics_list]

plt.bar(x - width, acc_vals, width, label='准确率')
plt.bar(x, rec_vals, width, label='召回率')
plt.bar(x + width, f1_vals, width, label='F1')
plt.xticks(x, TARGET_ATTRS, rotation=45, ha='right')
plt.ylim(0, 1)
plt.title('各属性评估指标对比')
plt.legend()
plt.tight_layout()
plt.savefig("./evaluation_results/attributes_comparison.png")
plt.show()

print("✅ 所有图表已生成并保存到 ./evaluation_results/")